In [ ]:
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns

sns.set_theme(style='whitegrid')

date_columns = ['last_scraped', 'calendar_last_scraped', 'first_review', 'last_review']
listings = pd.read_csv('data/listings_clean.csv', parse_dates=date_columns)

print(listings.shape)

## Lab 7 - Part 1
### Ex 1

In [ ]:
# pure matplotlib
fig, ax = plt.subplots()
ax.hist(listings['number_of_reviews'], bins=50)

plt.show()

# seaborn
fig, ax = plt.subplots()

sns.histplot(data=listings, x='number_of_reviews', bins=50, ax=ax)

ax.set_title('Most listings have very few reviews')
ax.set_xlabel('Number of reviews')
ax.set_ylabel('Number of listings')

plt.show()

#### data better represented in seaborn with labels and title.
### Ex 2

In [ ]:
num_reviews = listings['number_of_reviews']

q1 = num_reviews.quantile(0.25) 
q3 = num_reviews.quantile(0.75) 
iqr = q3 - q1

upper = q3 + 1.5 * iqr
mean = num_reviews.mean().round(2)
median = num_reviews.median()
above_upper = listings[num_reviews > upper]

print(upper, mean, median)

fig, ax = plt.subplots()

sns.histplot(data=listings, x='number_of_reviews', bins=50, ax=ax)

ax.axvline(median, color='black', linestyle='--', label=f'median {median:,.0f}')
ax.axvline(mean, color='red', linestyle='--', label=f'mean {mean:,.0f}')
ax.axvline(upper, color='gray', linestyle=':', label=f'upper fence {upper:,.0f}')

ax.set_title('The mean is pulled to the right by the long tail')
ax.set_xlabel('Number of reviews')
ax.set_ylabel('Number of listings')
ax.legend()

plt.show()

#### upper fence furthest to the right. again because of the long right tail.
### Ex 3

In [ ]:
listings['reviews_outlier'] = listings['number_of_reviews'] > upper
n_outlier = listings['reviews_outlier'].isin([True]).sum()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(data=listings, x='price', bins=50, ax=axes[0])
sns.histplot(data=listings[~listings['reviews_outlier']], x='number_of_reviews', bins=50, ax=axes[1])

axes[0].set_title('All listings')
axes[1].set_title(f'Without the {n_outlier} outliers')

for ax in axes:
    ax.set_xlabel('Number of reviews')
    ax.set_ylabel('Number of listings')

plt.tight_layout()

plt.show()

#### 582 outliers. The right chart makes the x-axis much narrower and you can more clearly see the distrubution.
#### It is still much alike with a slope tail to the right and majority at the extreme 0 or close to 0.
### Ex 4

In [ ]:
cut_off = 300
two_types = listings['room_type'].isin(['Entire home/apt', 'Private room'])
over_300 = listings['number_of_reviews'] > cut_off
n_excl = listings[two_types & over_300]['id'].count()

e_median = listings[listings['room_type'] == 'Entire home/apt']['number_of_reviews'].median()
e_mean = listings[listings['room_type'] == 'Entire home/apt']['number_of_reviews'].mean()
p_median = listings[listings['room_type'] == 'Private room']['number_of_reviews'].median()
p_mean = listings[listings['room_type'] == 'Private room']['number_of_reviews'].mean()

# 90 listings are excluded in the chart
print('n excluded', n_excl)

fig, ax = plt.subplots(figsize=(8, 5))

sns.boxplot(data=listings[two_types], x= 'room_type', y='number_of_reviews', ax=ax)

ax.set_ylim(0,cut_off)
ax.set_title(
    'Private room has higher mean and median'+
    f'\n{n_excl} listings have above {cut_off} reviews and are excluded')
ax.set_xlabel('')
ax.set_ylabel('Number of reviews')

ax.axhline(p_median, color='black', linestyle='--', label=f'p_median {p_median:,.0f}')
ax.axhline(p_mean, color='red', linestyle='--', label=f'p_mean {p_mean:,.0f}')
ax.axhline(upper, color='gray', linestyle=':', label=f'upper fence {upper:,.0f}')

ax.axhline(e_median, color='green', linestyle='--', label=f'e_median {e_median:,.0f}')
ax.axhline(e_mean, color='orange', linestyle='--', label=f'e_mean {e_mean:,.0f}')
ax.legend()

plt.show()

#### Now that's a chart.
### Ex 5

In [ ]:
# a)
order = listings['room_type'].value_counts().index

fig, ax = plt.subplots(figsize=(8, 5))

sns.countplot(data=listings, x='room_type', order=order, ax=ax)
ax.bar_label(ax.containers[0])

ax.set_title('Most listings are by far Entire home/apt')
ax.set_xlabel('')
ax.set_ylabel('Number of listings')


plt.tight_layout()
plt.show()

# b)
by_area = (
    listings.groupby('neighbourhood_cleansed')['number_of_reviews']
    .agg(['median', 'count'])
    .sort_values('median', ascending=False)
    .reset_index()
)
by_area['label'] = by_area['neighbourhood_cleansed'] + '(' + by_area['count'].astype(str) + ')'

print(by_area.head(3))

fif, ax = plt.subplots(figsize=(8, 5))

sns.barplot(data=by_area, x='median', y='label', ax=ax)

ax.set_title('Median number of reviews (the number of listings in brackets)')
ax.set_xlabel('Median number of reviews')
ax.set_ylabel('')

plt.tight_layout()
plt.show()

#### Just look at those bars. Majestic.
#### As we knew overwhelmingly majority entire home/apt, some private room. Rest pretty much a few of each.
### Ex 6

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

sns.scatterplot(data=listings[two_types], x='availability_365', y='number_of_reviews', 
                hue='room_type', alpha=0.5, ax=ax)

ax.set_ylim(0, 300)

ax.set_title('Entire homes are above private rooms for the same number of guests,'+
            f'\nexludes {n_excl} listings over {cut_off} reviews.')
ax.set_xlabel('Availability 365')
ax.set_ylabel('Number of reviews')

plt.show()

#### The dots. oooh the dots.
#### It is evenly spread along the whole x-axis and towards less reviews. No clear trend / pattern.
### Ex 7

In [ ]:
# a)
cols = ['number_of_reviews', 'price', 'accommodates', 'bedrooms', 'availability_365', 
        'review_scores_rating', 'minimum_nights', 'days_since_review']
corr = listings[cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))

sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, ax=ax)

ax.set_title('How the columns move together')

plt.tight_layout()
plt.show()

# b)
listings['last_review']
per_year = listings['last_review'].dt.year.value_counts().sort_index()

fig, ax = plt.subplots(figsize=(9,4))
sns.lineplot(x=per_year.index, y=per_year.values, marker='o', markers=True, ax= ax)

ax.set_title('Number of listings by the year of last review')
ax.set_xlabel('Year of the last review')
ax.set_ylabel('Number of listings')

plt.show()

#### We last written review is heavily weighted towards present day. As expected.
#### It tails off to the left. Low activity, long time booked or inactive listings.
### Ex 8

In [ ]:
rating_two_types = listings[two_types].groupby('room_type')['review_scores_rating'].median()

fig, axes = plt.subplots(1, 2, figsize= (11, 4))

for ax in axes:
    sns.barplot(x=['Entire home/apt', 'Private room'], y=rating_two_types, ax= ax)
    ax.set_ylabel('Median review score rating')
    ax.bar_label(ax.containers[0])

axes[0].set_title('Axis from 0')
axes[1].set_title('Axis from 4.88')
axes[1].set_ylim(4.88, 4.93)

plt.tight_layout()
plt.show()

#### You can easily make a miniscule difference seem significant by your choice of scale / zoom.